In [1]:
!pip install --upgrade nltk==3.9.1

In [2]:
import os
import nltk
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report
from nltk.tag import hmm
import numpy as np
from tqdm import tqdm
import pandas as pd

In [3]:
print(nltk.__version__)
print(np.__version__)

3.9.1
1.26.4


# Train English POS Tagger

In [ ]:
#tagged_sentences = nltk.corpus.treebank.tagged_sents(tagset='universal') #loading corpus
#tagged_sentences

In [ ]:
#traindataset , testdataset = train_test_split(tagged_sentences, shuffle=True, test_size=0.2) #Splitting test and train dataset

In [ ]:
#print("Train Data Length: ",len(traindataset))
#print("Test Data Length: ",len(testdataset))

## Approach 1

In [ ]:
##Training HMMModel
#HmmModel = nltk.HiddenMarkovModelTagger.train(traindataset)

##Correct labels from test dataset
#correct_labels = [tag for sentences in testdataset for word, tag in sentences]

##predicting labels of testdataset
#predicted_labels=[]
#for sentences in testdataset:
#    predicted_labels += [tag for _, tag in HmmModel.tag([word for word, _ in sentences])]

In [ ]:
#HmmModel.tag("Chicago is the birthplace of Ginny .".split())

In [ ]:
##Classification report
#print (classification_report(correct_labels, predicted_labels))

## Approach 2

In [ ]:
#trainer = hmm.HiddenMarkovModelTrainer()
#tagger = trainer.train_supervised(traindataset)

In [ ]:
#tagger._transitions

In [ ]:
#print(tagger.tag("Chicago is the birthplace of Ginny .".split()))

In [ ]:
#tagger.evaluate(testdataset)

# Train Konkani POS Tagger

## Collect Data

In [4]:
## Use this for 1L sentences
folder_path = '/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

100

In [5]:
if "kon_agriculture_set1.txt" in txt_files:
    txt_files.remove("kon_agriculture_set1.txt")

print(len(txt_files))

99


### Format the data

In [6]:
taggedData = []
for file in txt_files:
    with open("/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)
    
allTaggedData = [item for sublist in taggedData for item in sublist]
len(allTaggedData)

99091

In [7]:
allTaggedData = list(set(allTaggedData))
len(allTaggedData)

98986

## Clean The data

In [8]:
def clean_sentences(sentences): 
    cleaned_sentences = []
    for sentence in sentences:
        # Remove occurrences of '(\RD_PUNC' and ')\RD_PUNC'
        cleaned_sentence = sentence.replace('(\RD_PUNC', '').replace(')\RD_PUNC', '')
        cleaned_sentence = sentence.replace('\\RD_PUNC', '').replace(')\RD_PUNC', '')
        cleaned_sentence = cleaned_sentence.replace('  ', '')
        cleaned_sentences.append(cleaned_sentence)
    return cleaned_sentences

# Clean the sentences
allTaggedData_cleaned = clean_sentences(allTaggedData)

In [9]:
def transform_data(data):
    transformed_data = []
    for sentence in data:
        word_pos_pairs = sentence.split(' ')  # Split sentence into word-POS pairs
        tuples = [tuple(word_pos.split('\\')) for word_pos in word_pos_pairs]  # Create tuples
        transformed_data.append(tuples)
    return transformed_data


final_tagged_data = transform_data(allTaggedData_cleaned)
len(final_tagged_data)

98986

In [10]:
traindataset , testdataset = train_test_split(final_tagged_data, shuffle=True, test_size=0.05)
print("Train Data Length: ",len(traindataset))
print("Test Data Length: ",len(testdataset))

Train Data Length:  94036
Test Data Length:  4950


In [11]:
def remove_tuples_with_lengths(sentences, lengths=[1, 3, 4]):
    cleaned_sentences = []
    found = False 

    for sentence in sentences:
        for length in lengths:
            # Check if any tuple in the sentence has the current length
            if any(len(token) == length for token in sentence):
                found = True
                #print(f"Found tuples with length {length} in sentence: {sentence}")
        
        # Filter out tokens whose length is in the specified lengths list
        cleaned_sentence = [token for token in sentence if len(token) not in lengths]
        cleaned_sentences.append(cleaned_sentence)
    
    if not found:
        print(f"No tuples with lengths {lengths} found in any sentence.")
    
    return cleaned_sentences

In [12]:
traindataset = remove_tuples_with_lengths(traindataset)
testdataset = remove_tuples_with_lengths(testdataset)
print("Train Data Length: ",len(traindataset))
print("Test Data Length: ",len(testdataset))

Train Data Length:  94036
Test Data Length:  4950


In [13]:
traindataset_removeBlank = [sent for sent in traindataset if sent]
testdataset_removeBlank = [sent for sent in testdataset if sent]
print(len(traindataset_removeBlank))
print(len(testdataset_removeBlank))

93752
4931


## Train a Model: Approach 1

In [ ]:
from nltk.probability import LidstoneProbDist
from nltk.probability import LaplaceProbDist

from nltk.tag import UnigramTagger

In [ ]:
unigram_tagger = UnigramTagger(traindataset_removeBlank)

In [ ]:
trainer = hmm.HiddenMarkovModelTrainer()
tagger = trainer.train_supervised(traindataset_removeBlank, estimator=lambda fd, bins: LidstoneProbDist(fd, 0.1, bins))
#tagger = trainer.train_supervised(traindataset_removeBlank, estimator=lambda fd, bins: LaplaceProbDist(fd, bins))

In [ ]:
#train_accuracy_1 = tagger.accuracy(traindataset_removeBlank)
#print("Train Accuracy Approach 1: ",train_accuracy_1)

In [ ]:
from tqdm import tqdm

# Extract words and true tags from the dataset
correct = 0
total = 0

for sent in tqdm(traindataset_removeBlank, desc="Calculating Training Accuracy", unit="sentence"):
    words, true_tags = zip(*sent)
    predicted_tags = [tag for _, tag in tagger.tag(words)]

    # Count correct predictions
    correct += sum(1 for t, p in zip(true_tags, predicted_tags) if t == p)
    total += len(true_tags)

# Compute accuracy
train_accuracy_1 = correct / total if total > 0 else 0.0
print(f"Training Accuracy: {train_accuracy_1:.2%}")

In [ ]:
predicted_tags_train_1 = []
true_tags_train_1 = []

for sentence in traindataset_removeBlank:
    words = [word for word, _ in sentence]
    true_tags_train_1.extend([tag for _, tag in sentence])
    tagged_sentence = tagger.tag(words)
    predicted_tags_train_1.extend([tag for _, tag in tagged_sentence])

In [ ]:
# Calculate precision, recall, and F1-score
train_precision_1, train_recall_1, train_fscore_1, _ = precision_recall_fscore_support(true_tags_train_1, predicted_tags_train_1, average='weighted')

#print(tagger._transitions) 
print("Evaluation Score For Training Data:")
print("Accuracy: ", train_accuracy_1)
print("Precision: ", train_precision_1)
print("Recall: ", train_recall_1)
print("F1-score: ", train_fscore_1)
print (classification_report(true_tags_train_1, predicted_tags_train_1))

In [ ]:
#test_accuracy_1 = tagger.accuracy(testdataset_removeBlank)
#print("Test Accuracy Approach 1: ",test_accuracy_1)

In [ ]:
# Extract words and true tags from the dataset
correct = 0
total = 0

for sent in tqdm(testdataset_removeBlank, desc="Calculating Test Accuracy", unit="sentence"):
    words, true_tags = zip(*sent)
    predicted_tags = [tag for _, tag in tagger.tag(words)]

    # Count correct predictions
    correct += sum(1 for t, p in zip(true_tags, predicted_tags) if t == p)
    total += len(true_tags)

# Compute accuracy
test_accuracy_1 = correct / total if total > 0 else 0.0
print(f"Test Accuracy: {test_accuracy_1:.2%}")

In [ ]:
predicted_tags_test_1 = []
true_tags_test_1 = []

for sentence in testdataset_removeBlank:
    #print(sentence)
    words = [word for word, _ in sentence]
    true_tags_test_1.extend([tag for _, tag in sentence])
    tagged_sentence = tagger.tag(words)
    #print(tagged_sentence)
    predicted_tags_test_1.extend([tag for _, tag in tagged_sentence])

In [ ]:
#print(true_tags_test_1)

In [ ]:
# Calculate precision, recall, and F1-score
test_precision_1, test_recall_1, test_fscore_1, _ = precision_recall_fscore_support(true_tags_test_1, predicted_tags_test_1, average='weighted')

#print(tagger._transitions) 
print("Evaluation Score For Testing Data:")
print("Accuracy: ", test_accuracy_1)
print("Precision: ", test_precision_1)
print("Recall: ", test_recall_1)
print("F1-score: ", test_fscore_1)
print (classification_report(true_tags_test_1, predicted_tags_test_1))

In [ ]:
# वारें\N_NN मुखा\N_NN वयल्यान\N_NST येतालें\V_VM_VF .\RD_PUNC
# वारें मुखा वयल्यान येतालें .
# आडसरां\N_NN इतलीं\QT_QTF की\CC_CCS_UT उदकाची\N_NN गरजूच\RB पडना\V_VM_VF .\RD_PUNC 
# आडसरां इतलीं की उदकाची गरजूच पडना .
def preprocess_text(text):
    return text.split()  # Split the text into words

text = "हीं कर्जां भरपाक मागीर आनीक हप्ते ."  
tokens = preprocess_text(text)
tagged = tagger.tag(tokens)
tagged

In [ ]:
import pickle
import dill

In [ ]:
# Save the tagger using dill
with open("hmm_uniram_tagger.pkl", "wb") as f:
    dill.dump(tagger, f)

In [ ]:
# Load the tagger
with open("/kaggle/working/hmm_uniram_tagger.pkl", "rb") as f:
    loaded_tagger = dill.load(f)

In [ ]:
text = "हीं कर्जां भरपाक मागीर आनीक हप्ते ."  
tokens = preprocess_text(text)
tagged = loaded_tagger.tag(tokens)
tagged

In [ ]:
def predict_pos(text):
    tokens = preprocess_text(text)
    tagged = loaded_tagger.tag(tokens)
    return tagged

predict_pos("हीं कर्जां भरपाक मागीर आनीक हप्ते .")

In [ ]:
testData = []
file_path = "/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/kon_agriculture_set1.txt"

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

content = content.split("\n")[1:]  # Skip the first line
testDataFile = [line.split('\t')[1] for line in content if line and 'Value' not in line.split('\t')[1]]
testData.append(testDataFile)

testTaggedData = [item for sublist in testData for item in sublist]
print(testTaggedData[:2])

In [ ]:
print(len(testTaggedData))
ref_sentences = testTaggedData[:75]
print(len(ref_sentences))

In [ ]:
df_test = pd.DataFrame(ref_sentences, columns=['actual_tags'])
df_test.info()

In [ ]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
df_test["sentence"], df_test["labels"] = zip(*df_test["actual_tags"].map(preprocess_sentence))
df_test.info()

In [ ]:
def joinSen(words):
    sentence = " ".join(words)
    return sentence


df_test["joined_sent"] = df_test["sentence"].apply(joinSen)
df_test.head()

In [ ]:
df_test["output"] = df_test["joined_sent"].apply(predict_pos)
df_test.head()

In [ ]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

In [ ]:
df_test["joined_output"] = df_test["output"].apply(makeTagedSentences)
df_test.head()

In [ ]:
df_test.drop(["output", "sentence", "labels"], axis=1, inplace=True)
df_test

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [ ]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['actual_tags'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_test)
mismatched_indices

In [ ]:
for ind in mismatched_indices:
    df_test = df_test.drop(ind)

In [ ]:
# Apply function to each row
df_test[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_test.apply(lambda row: evaluate_tagging(row["actual_tags"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_test.head()

In [ ]:
df_test.to_excel("test_hmm_ungram_75_output.xlsx", index=False)

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_test["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_test["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

In [ ]:
df_ref = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/ref_sentences_Mannual.xlsx')
df_ref.head()

In [ ]:
df_ref = df_ref[:-1]

In [ ]:
df_ref.tail()

In [ ]:
df_ref["output"] = df_ref["sentences"].apply(predict_pos)
df_ref.head()

In [ ]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

In [ ]:
df_ref["joined_output"] = df_ref["output"].apply(makeTagedSentences)
df_ref.head()

In [ ]:
df_ref.drop(["output"], axis=1, inplace=True)
df_ref

In [ ]:

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['ref_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_ref)
mismatched_indices

In [ ]:
for ind in mismatched_indices:
    df_ref = df_ref.drop(ind)

In [ ]:
# Apply function to each row
df_ref[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_ref.apply(lambda row: evaluate_tagging(row["ref_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_ref.head()

In [ ]:
df_ref.to_excel("test_hmm_unigram_100_manual_output.xlsx", index=False)

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_ref["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_ref["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

In [ ]:
df_anwani = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/anwani_ref_sentences.xlsx')
df_anwani.head()

In [ ]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(predict_pos)
df_anwani.head()

In [ ]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["Unnamed: 3"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

In [ ]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
mismatched_indices

In [ ]:
for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

In [ ]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

In [ ]:
df_anwani.to_excel("test_hmm_unigram_100_anwani_output.xlsx", index=False)

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

## Train a Model: Approach 2

In [14]:
#Training HMMModel
HmmModel = nltk.HiddenMarkovModelTagger.train(traindataset_removeBlank)

In [ ]:
# labels from train dataset
train_correct_labels_2 = [tag for sentences in traindataset_removeBlank for word, tag in sentences]

In [ ]:
#predicting labels of testdataset
train_predicted_labels_2=[]
for sentences in traindataset_removeBlank:
    train_predicted_labels_2 += [tag for _, tag in HmmModel.tag([word for word, _ in sentences])]

In [ ]:
train_accuracy_2 = HmmModel.accuracy(traindataset_removeBlank)
print(train_accuracy_2)

In [ ]:
# Calculate precision, recall, and F1-score
train_precision_2, train_recall_2, train_fscore_2, _ = precision_recall_fscore_support(train_correct_labels_2, train_predicted_labels_2, average='weighted')

#print(tagger._transitions) 
print("Evaluation Score For Training Data:")
print("Accuracy: ", train_accuracy_2)
print("Precision: ", train_precision_2)
print("Recall: ", train_recall_2)
print("F1-score: ", train_fscore_2)
print (classification_report(train_correct_labels_2, train_predicted_labels_2))

In [ ]:
#Correct labels from test dataset
test_correct_labels_2 = [tag for sentences in testdataset_removeBlank for word, tag in sentences]

In [ ]:
#predicting labels of testdataset
test_predicted_labels_2=[]
for sentences in testdataset_removeBlank:
    test_predicted_labels_2 += [tag for _, tag in HmmModel.tag([word for word, _ in sentences])]

In [ ]:
test_accuracy_2 = HmmModel.accuracy(testdataset_removeBlank)
print(test_accuracy_2)

In [ ]:
# Calculate precision, recall, and F1-score
test_precision_2, test_recall_2, test_fscore_2, _ = precision_recall_fscore_support(test_correct_labels_2, test_predicted_labels_2, average='weighted')

#print(tagger._transitions) 
print("Evaluation Score For Testing Data:")
print("Accuracy: ", test_accuracy_2)
print("Precision: ", test_precision_2)
print("Recall: ", test_recall_2)
print("F1-score: ", test_fscore_2)
print (classification_report(test_correct_labels_2, test_predicted_labels_2))

In [ ]:
# वारें\N_NN मुखा\N_NN वयल्यान\N_NST येतालें\V_VM_VF .\RD_PUNC
# वारें मुखा वयल्यान येतालें .
# आडसरां\N_NN इतलीं\QT_QTF की\CC_CCS_UT उदकाची\N_NN गरजूच\RB पडना\V_VM_VF .\RD_PUNC 
# आडसरां इतलीं की उदकाची गरजूच पडना .
def preprocess_text(text):
    return text.split()  # Split the text into words

text = "हीं कर्जां भरपाक मागीर आनीक हप्ते ."  
tokens = preprocess_text(text)
tagged = HmmModel.tag(tokens)
tagged

In [ ]:
# Save the tagger using dill
with open("Hmm_bigram_Model.pkl", "wb") as f:
    dill.dump(HmmModel, f)

In [ ]:
# Load the tagger
with open("/kaggle/working/Hmm_bigram_Model.pkl", "rb") as f:
    loaded_bitagger = dill.load(f)

In [ ]:
text = "हीं कर्जां भरपाक मागीर आनीक हप्ते ."  
tokens = preprocess_text(text)
tagged = loaded_bitagger.tag(tokens)
tagged

In [ ]:
def predict_pos(text):
    tokens = preprocess_text(text)
    tagged = loaded_bitagger.tag(tokens)
    return tagged

predict_pos("हीं कर्जां भरपाक मागीर आनीक हप्ते .")

In [ ]:
testData = []
file_path = "/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/kon_agriculture_set1.txt"

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

content = content.split("\n")[1:]  # Skip the first line
testDataFile = [line.split('\t')[1] for line in content if line and 'Value' not in line.split('\t')[1]]
testData.append(testDataFile)

testTaggedData = [item for sublist in testData for item in sublist]
print(testTaggedData[:2])

In [ ]:
print(len(testTaggedData))
ref_sentences = testTaggedData[:75]
print(len(ref_sentences))

In [ ]:
df_test = pd.DataFrame(ref_sentences, columns=['actual_tags'])
df_test.info()

In [ ]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
df_test["sentence"], df_test["labels"] = zip(*df_test["actual_tags"].map(preprocess_sentence))
df_test.info()

In [ ]:
def joinSen(words):
    sentence = " ".join(words)
    return sentence


df_test["joined_sent"] = df_test["sentence"].apply(joinSen)
df_test.head()

In [ ]:
df_test["output"] = df_test["joined_sent"].apply(predict_pos)
df_test.head()

In [ ]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

In [ ]:
df_test["joined_output"] = df_test["output"].apply(makeTagedSentences)
df_test.head()

In [ ]:
df_test.drop(["output", "sentence", "labels"], axis=1, inplace=True)
df_test

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [ ]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['actual_tags'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_test)
mismatched_indices

In [ ]:
for ind in mismatched_indices:
    df_test = df_test.drop(ind)

In [ ]:
# Apply function to each row
df_test[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_test.apply(lambda row: evaluate_tagging(row["actual_tags"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_test.head()

In [ ]:
df_test.to_excel("test_hmm_bigram_75_output.xlsx", index=False)

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_test["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_test["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

In [ ]:
df_ref = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/ref_sentences_Mannual.xlsx')
df_ref.head()

In [ ]:
df_ref.tail()

In [ ]:
df_ref = df_ref[:-1]

In [ ]:
df_ref["output"] = df_ref["sentences"].apply(predict_pos)
df_ref.head()

In [ ]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

In [ ]:
df_ref["joined_output"] = df_ref["output"].apply(makeTagedSentences)
df_ref.head()

In [ ]:
df_ref.drop(["output"], axis=1, inplace=True)
df_ref

In [ ]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['ref_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_ref)
mismatched_indices

In [ ]:
for ind in mismatched_indices:
    df_ref = df_ref.drop(ind)

In [ ]:
# Apply function to each row
df_ref[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_ref.apply(lambda row: evaluate_tagging(row["ref_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_ref.head()

In [ ]:
df_ref.to_excel("test_hmm_bigram_100_manual_output.xlsx", index=False)

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_ref["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_ref["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

In [ ]:
df_anwani = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/anwani_ref_sentences.xlsx')
df_anwani.head()

In [ ]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(predict_pos)
df_anwani.head()

In [ ]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["Unnamed: 3"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

In [ ]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
mismatched_indices

In [ ]:
for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

In [ ]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

In [ ]:
df_anwani.to_excel("test_hmm_bigram_100_anwani_output.xlsx", index=False)

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

## Train a Model: Approach 3

In [ ]:
import nltk
from nltk.tag import hmm
from sklearn.metrics import precision_recall_fscore_support, classification_report

In [ ]:
# Ensure compatibility with the latest NLTK version
nltk.download('averaged_perceptron_tagger')

In [ ]:
# Train HMM model (default: bigram)
trainer = hmm.HiddenMarkovModelTrainer()
HmmModel = trainer.train(traindataset_removeBlank)

In [ ]:
# Helper functions
def get_labels(dataset):
    return [tag for sentence in dataset for _, tag in sentence]

def get_predictions(dataset, model):
    return [tag for sentence in dataset for _, tag in model.tag([word for word, _ in sentence])]

In [ ]:
# Evaluate
train_correct_labels = get_labels(traindataset_removeBlank)
train_predicted_labels = get_predictions(traindataset_removeBlank, HmmModel)
train_accuracy = HmmModel.accuracy(traindataset_removeBlank)

In [ ]:
# Metrics
train_precision, train_recall, train_fscore, _ = precision_recall_fscore_support(
    train_correct_labels, train_predicted_labels, average='weighted'
)

In [ ]:
# Results
print("\nEvaluation Score for Training Data:")
print(f"Accuracy: {train_accuracy:.2%}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall: {train_recall:.4f}")
print(f"F1-score: {train_fscore:.4f}")
print(classification_report(train_correct_labels, train_predicted_labels))

In [ ]:
# Testing Evaluation
test_correct_labels = get_labels(testdataset_removeBlank)
test_predicted_labels = get_predictions(testdataset_removeBlank, HmmModel)

In [ ]:
test_accuracy = HmmModel.accuracy(testdataset_removeBlank)

In [ ]:
# Calculate precision, recall, and F1-score
test_precision, test_recall, test_fscore, _ = precision_recall_fscore_support(
    test_correct_labels, test_predicted_labels, average='weighted'
)

print("\nEvaluation Score for Testing Data:")
print(f"Accuracy: {test_accuracy:.2%}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1-score: {test_fscore:.4f}")
print(classification_report(test_correct_labels, test_predicted_labels))

In [ ]:
def preprocess_text(text):
    return text.split()  # Split the text into words

text = "हीं कर्जां भरपाक मागीर आनीक हप्ते ."  
tokens = preprocess_text(text)
tagged = HmmModel.tag(tokens)
tagged

In [ ]:
def predict_pos(text):
    tokens = preprocess_text(text)
    tagged = HmmModel.tag(tokens)
    return tagged

predict_pos("हीं कर्जां भरपाक मागीर आनीक हप्ते .")

In [30]:
df_anwani = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/anwani_ref_sentences.xlsx')
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN


In [31]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(predict_pos)
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3,output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN,"[(आयकलां, V_VM_VF), (?, N_NN), (लागीं, N_NST),..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN,"[(दोन, QT_QTC), (तीन, QT_QTC), (दिसांनी, N_NN)..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN,"[(आनी, CC_CCD), (जाली, V_VM_VF), (ना, V_VM_VF)..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN,"[(हांव, PR_PRP), (अशें, DM_DMD), (आसा, V_VM_VF..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN,"[(पूण, CC_CCS), (हांगा, N_NST), (रावनय, N_NN),..."


In [32]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

'सहस्त्रधारा\\N_NNP ही\\DM_DMD देहरादूनाची\\N_NNP पळोवपाची\\V_VM_VNF मुखेल\\JJ सुवात\\N_NN .\\RD_PUNC'

In [33]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["Unnamed: 3"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\N_NN लागीं\N_NST ना\RP_NEG आम...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\V_VM_VF जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...


In [34]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
mismatched_indices

[48, 49]

In [35]:
for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98 entries, 0 to 99
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences_cleaned     98 non-null     object
 1   tagged_sentences      98 non-null     object
 2   joined_output         98 non-null     object
 3   ref_sentences_length  98 non-null     int64 
 4   joined_sent_length    98 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 4.6+ KB


In [36]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [37]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\N_NN लागीं\N_NST ना\RP_NEG आम...,14,14,0.571429,0.478571,0.571429,0.500000,"[V_VM_VF, RD_PUNC, N_NST, RP_NEG, DM_DMD, N_NN...","[V_VM_VF, N_NN, N_NST, RP_NEG, PR_PRP, N_NN, N..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,11,11,0.818182,0.709091,0.818182,0.750000,"[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_...","[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, N_N..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\V_VM_VF जाल्यार\CC_...,22,22,0.636364,0.660714,0.636364,0.597383,"[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR...","[CC_CCD, V_VM_VF, V_VM_VF, CC_CCS, N_NN, PR_PR..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,17,17,0.823529,0.784314,0.823529,0.800000,"[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM...","[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...,15,15,0.733333,0.755556,0.733333,0.730303,"[CC_CCS, N_NST, V_VM_VNF, V_VM_VNF, RP_NEG, RD...","[CC_CCS, N_NST, N_NN, V_VM_VNF, RP_NEG, RD_UNK..."


In [38]:
df_anwani.to_excel("test_hmm_trigram_100_anwani_output.xlsx", index=False)

In [39]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        13
      CC_CCS       0.58      0.91      0.71        43
      DM_DMD       0.90      0.87      0.88        52
      DM_DMI       1.00      0.70      0.82        10
      DM_DMQ       1.00      0.13      0.24        15
      DM_DMR       0.00      0.00      0.00         0
          JJ       0.26      0.58      0.36        12
         NST       0.00      0.00      0.00         1
        N_NN       0.51      0.96      0.66       201
       N_NNP       0.00      0.00      0.00         1
       N_NST       0.87      0.83      0.85        78
      PR_PRF       1.00      1.00      1.00         1
      PR_PRI       0.75      0.27      0.40        11
      PR_PRL       0.00      0.00      0.00         0
      PR_PRP       0.95      0.93      0.94       131
      PR_PRQ       0.48      0.55      0.51        20
         PSP       0.77      0.53      0.62        38
  

In [18]:
import nltk
from nltk.tag import UnigramTagger, BigramTagger, TrigramTagger
from sklearn.metrics import precision_recall_fscore_support, classification_report

In [20]:
# Unigram tagger
unigram_tagger = UnigramTagger(traindataset_removeBlank)

# Bigram tagger with unigram backoff
bigram_tagger = BigramTagger(traindataset_removeBlank, backoff=unigram_tagger)

# Trigram tagger with bigram backoff
trigram_tagger = TrigramTagger(traindataset_removeBlank, backoff=bigram_tagger)

In [21]:
train_predicted_tags = []
train_true_tags = []

for sentence in traindataset_removeBlank:
    words = [word for word, _ in sentence]
    true_tags = [tag for _, tag in sentence]
    predicted_tags = [tag for _, tag in trigram_tagger.tag(words)]
    
    train_true_tags.extend(true_tags)
    train_predicted_tags.extend(predicted_tags)

# Accuracy
train_accuracy = trigram_tagger.evaluate(traindataset_removeBlank)

# Precision, Recall, F1
train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
    train_true_tags, train_predicted_tags, average='weighted', zero_division=0
)

print("\n📊 Evaluation on Training Data:")
print(f"Accuracy:  {train_accuracy:.2%}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-score:  {train_f1:.4f}")
print(classification_report(train_true_tags, train_predicted_tags, zero_division=0))

/tmp/ipykernel_65/3101495156.py:13: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  train_accuracy = trigram_tagger.evaluate(traindataset_removeBlank)



📊 Evaluation on Training Data:
Accuracy:  93.04%
Precision: 0.9304
Recall:    0.9304
F1-score:  0.9300
              precision    recall  f1-score   support

                   1.00      0.60      0.75        10
      CC_CCD       1.00      0.99      0.99     31651
      CC_CCS       0.87      0.93      0.90     21173
   CC_CCS_UT       1.00      0.02      0.04        88
    CC_CC_UT       0.49      0.08      0.14       503
       CC_UT       0.46      0.10      0.16       469
      DM_DMD       0.86      0.90      0.88     40577
      DM_DMI       0.82      0.77      0.79      2047
      DM_DMQ       0.68      0.52      0.59       822
      DM_DMR       0.62      0.56      0.59      5851
      DN_DMD       1.00      1.00      1.00         1
          JJ       0.90      0.89      0.89     68633
         JJ,       1.00      1.00      1.00         1
       N-NNP       1.00      1.00      1.00         1
          NN       1.00      0.25      0.40         4
       NN_NN       1.00      1.

In [27]:
test_predicted_tags = []
test_true_tags = []

for sentence in testdataset_removeBlank:
    words = [word for word, _ in sentence]
    true_tags = [tag for _, tag in sentence]
    predicted = trigram_tagger.tag(words)
    
    for (word, pred_tag), true_tag in zip(predicted, true_tags):
        final_tag = pred_tag if pred_tag is not None else 'N_NN'  # Replace None with N_NN
        test_predicted_tags.append(final_tag)
        test_true_tags.append(true_tag)

# Accuracy
test_accuracy = sum(1 for p, t in zip(test_predicted_tags, test_true_tags) if p == t) / len(test_true_tags)

# Precision, Recall, F1
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_true_tags, test_predicted_tags, average='weighted', zero_division=0
)

print("\n📊 Evaluation on Test Data:")
print(f"Accuracy:  {test_accuracy:.2%}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(classification_report(test_true_tags, test_predicted_tags, zero_division=0))


📊 Evaluation on Test Data:
Accuracy:  84.78%
Precision: 0.8459
Recall:    0.8478
F1-score:  0.8439
              precision    recall  f1-score   support

                   0.00      0.00      0.00         0
      CC_CCD       1.00      0.99      0.99      1533
      CC_CCS       0.83      0.91      0.87      1060
   CC_CCS_UT       0.00      0.00      0.00         6
    CC_CC_UT       0.38      0.09      0.14        34
       CC_UT       1.00      0.10      0.17        21
      DM_DMD       0.83      0.85      0.84      2134
      DM_DMI       0.61      0.60      0.61       116
      DM_DMQ       0.35      0.23      0.28        52
      DM_DMR       0.50      0.47      0.48       293
          JJ       0.78      0.73      0.76      3538
        N_NN       0.87      0.95      0.91     20689
       N_NNP       0.86      0.67      0.75      4253
       N_NNV       0.00      0.00      0.00        12
       N_NST       0.84      0.82      0.83      1624
      PR_PRC       0.76      0.76  

In [29]:
def preprocess_text(text):
    return text.split()  # Split the text into words
    
def predict_pos(text):
    tokens = preprocess_text(text)
    tagged = trigram_tagger.tag(tokens)
    
    # Replace None with 'N_NN'
    tagged = [(word, tag if tag is not None else 'N_NN') for word, tag in tagged]
    
    return tagged

predict_pos("हीं कर्जां भरपाक मागीर आनीक हप्ते .")

[('हीं', 'DM_DMD'),
 ('कर्जां', 'N_NN'),
 ('भरपाक', 'V_VM_VNF'),
 ('मागीर', 'N_NST'),
 ('आनीक', 'QT_QTF'),
 ('हप्ते', 'N_NN'),
 ('.', 'RD_UNK')]

# Data Preprocessing Functions

In [ ]:
def find_sentence_with_word(dataset, word):
    for i, sentence in enumerate(dataset):
        # Check if the word is in any of the tuples in the sentence
        if any(word == token.split('\\')[0] for token in sentence):
            return i  # return index of the sentence containing the word
    return -1  # return -1 if not found

# Example dataset
dataset = [
    'जाल्यार\\CC_CCS_UT तुमी\\PR_PRP दोळ्यांच्या\\N_NN पिडां\\N_NN पसुनूय\\PSP वाटावतले\\V_VM_VF .\\RD_PUNC',
    'लिवरपूल\\N_NNP विश्वविद्यालयाच्या\\N_NNP ऑफ्थाल्मोलॉजी\\N_NN विभागाचो\\N_NN अध्यक्ष\\N_NN आनी\\CC_CCD नामनेचो\\JJ नेत्ररोग\\N_NN तज्ञ\\N_NN डॉ\\N_NN .\\RD_PUNC इयान\\N_NNP ग्रियरसन\\N_NNP म्हणटा\\V_VM_VNF .\\RD_PUNC'
]

# Find the index of the sentence containing the word 'तुमी'
index = find_sentence_with_word(dataset, 'तुमी')

# Print result
if index != -1:
    print(f"Sentence containing 'तुमी' is at index {index}.")
else:
    print("Word 'तुमी' not found in any sentence.")

In [ ]:
# Verify the length of each tuple
for sentence in traindataset:
    for token in sentence:
        if len(token) != 2:
            raise ValueError("Incorrect tuple length in training data: {}".format(token))

In [ ]:
def find_sentence_with_tuple(dataset, tag):
    for i, sentence in enumerate(dataset):
        # Check if any tuple in the sentence matches the ('', '', tag) pattern
        if any(len(token) == 3 and token[2] == tag and token[0] == '' and token[1] == '' for token in sentence):
            return i  # return index of the sentence containing the tuple
    return -1  # return -1 if not found

# Find the index of the sentence containing the tuple ('', '', 'RD_PUNC')
index = find_sentence_with_tuple(traindataset, 'RD_PUNC')

# Print result
if index != -1:
    print(f"Sentence containing the tuple ('', '', 'RD_PUNC') is at index {index}.")
else:
    print("Tuple ('', '', 'RD_PUNC') not found in any sentence.")

In [ ]:
def find_sentence_with_tuple_length(dataset, length):
    for i, sentence in enumerate(dataset):
        # Check if any tuple in the sentence has the specified length
        if any(len(token) == length for token in sentence):
            return i  # return index of the sentence containing the tuple
    return -1  # return -1 if not found

# Find the index of the sentence containing a tuple of length 3
index = find_sentence_with_tuple_length(traindataset, 3)

# Print result
if index != -1:
    print(f"Sentence containing a tuple of length 3 is at index {index}.")
else:
    print("No sentence contains a tuple of length 3.")
